# Transfer Learning for SMS Spam Detection

## Overview

This notebook demonstrates transfer learning using pre-trained transformer models
for SMS spam classification. Transfer learning is particularly effective for
small datasets (~5000 SMS) as these models bring rich language understanding
from pre-training on massive corpora.

**Models evaluated:**
- **RoBERTa-base**: Robust optimized BERT variant, known for strong
  performance on text classification (target: >99% accuracy)
- **DistilBERT-base**: Distilled version of BERT, 40% smaller and 60% faster
  while retaining 97% of BERT's performance

**Objective:** Achieve high precision on spam class to minimize false
positives (legitimate messages marked as spam).

## 1. Setup and Imports

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from datasets import Dataset as HFDataset

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

### Device Detection and Reproducibility

In [ ]:
# Device detection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memory: {mem_gb:.2f} GB")

# Set seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

# Define paths
BASE_DIR = Path.cwd().parent  # Assumes notebook runs from notebooks/
DATA_DIR = BASE_DIR / 'data' / 'processed'
MODELS_DIR = BASE_DIR / 'models'
MODELS_DIR.mkdir(exist_ok=True)

print(f"\nData directory: {DATA_DIR}")
print(f"Models directory: {MODELS_DIR}")

## 2. Load Data

Load preprocessed train, validation, and test sets.

In [ ]:
# Load datasets
train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df = pd.read_csv(DATA_DIR / 'val.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

print("Dataset Statistics:")
print("=" * 60)
print(f"Train set: {len(train_df)} samples")
ham_train_pct = (train_df['label'] == 0).mean() * 100
spam_train_pct = (train_df['label'] == 1).mean() * 100
print(f"  - Ham: {(train_df['label'] == 0).sum()} ({ham_train_pct:.1f}%)")
print(f"  - Spam: {(train_df['label'] == 1).sum()} ({spam_train_pct:.1f}%)")
print()
print(f"Validation set: {len(val_df)} samples")
ham_val_pct = (val_df['label'] == 0).mean() * 100
spam_val_pct = (val_df['label'] == 1).mean() * 100
print(f"  - Ham: {(val_df['label'] == 0).sum()} ({ham_val_pct:.1f}%)")
print(f"  - Spam: {(val_df['label'] == 1).sum()} ({spam_val_pct:.1f}%)")
print()
print(f"Test set: {len(test_df)} samples")
ham_test_pct = (test_df['label'] == 0).mean() * 100
spam_test_pct = (test_df['label'] == 1).mean() * 100
print(f"  - Ham: {(test_df['label'] == 0).sum()} ({ham_test_pct:.1f}%)")
print(f"  - Spam: {(test_df['label'] == 1).sum()} ({spam_test_pct:.1f}%)")
print("=" * 60)

# Display sample messages
print("\nSample messages:")
print(train_df.head())

### Analyze Text Lengths for Tokenization

In [ ]:
# Calculate text lengths (words)
train_df['text_length'] = train_df['text'].str.split().str.len()

print("Text length statistics (words):")
print(train_df['text_length'].describe())
print(f"\n95th percentile: {train_df['text_length'].quantile(0.95):.0f} words")
print(f"99th percentile: {train_df['text_length'].quantile(0.99):.0f} words")

# Visualize
plt.figure(figsize=(10, 4))
plt.hist(train_df['text_length'], bins=50, edgecolor='black', alpha=0.7)
plt.axvline(train_df['text_length'].quantile(0.95), color='r', 
            linestyle='--', label='95th percentile')
plt.xlabel('Number of words')
plt.ylabel('Frequency')
plt.title('Distribution of SMS Text Lengths')
plt.legend()
plt.tight_layout()
plt.show()

# Set max_length based on data
MAX_LENGTH = int(train_df['text_length'].quantile(0.99) * 1.5)  # Words to tokens
MAX_LENGTH = min(MAX_LENGTH, 128)  # Cap at 128 for efficiency
print(f"\nUsing max_length={MAX_LENGTH} for tokenization")

## 3. RoBERTa Fine-tuning

RoBERTa (Robustly Optimized BERT Pretraining Approach) is an improved
version of BERT with better performance on downstream tasks.

### 3.1 Tokenizer and Dataset Preparation

In [ ]:
# Load RoBERTa tokenizer
roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

def tokenize_function(examples, tokenizer, max_length):
    """
    Tokenize text examples for transformer models.
    
    Args:
        examples: Dictionary with 'text' key containing list of texts
        tokenizer: HuggingFace tokenizer instance
        max_length: Maximum sequence length for tokenization
    
    Returns:
        Dictionary with tokenized inputs (input_ids, attention_mask)
    """
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors=None,  # Return lists for HF Dataset compatibility
    )

# Convert pandas to HuggingFace Dataset
train_dataset = HFDataset.from_pandas(train_df[['text', 'label']])
val_dataset = HFDataset.from_pandas(val_df[['text', 'label']])
test_dataset = HFDataset.from_pandas(test_df[['text', 'label']])

# Tokenize datasets
roberta_train = train_dataset.map(
    lambda x: tokenize_function(x, roberta_tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=['text']
)
roberta_val = val_dataset.map(
    lambda x: tokenize_function(x, roberta_tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=['text']
)
roberta_test = test_dataset.map(
    lambda x: tokenize_function(x, roberta_tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=['text']
)

# Set format for PyTorch
format_cols = ['input_ids', 'attention_mask', 'label']
roberta_train.set_format(type='torch', columns=format_cols)
roberta_val.set_format(type='torch', columns=format_cols)
roberta_test.set_format(type='torch', columns=format_cols)

print(f"Tokenized datasets:")
print(f"  Train: {len(roberta_train)} samples")
print(f"  Validation: {len(roberta_val)} samples")
print(f"  Test: {len(roberta_test)} samples")
print(f"\nSample tokenized input:")
print(roberta_train[0])

### 3.2 Model Initialization

In [ ]:
# Load RoBERTa model for sequence classification
roberta_model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=2,
    problem_type='single_label_classification'
)

# Move to device
roberta_model.to(device)

# Display model info
num_params = sum(p.numel() for p in roberta_model.parameters())
num_trainable = sum(p.numel() for p in roberta_model.parameters() if p.requires_grad)

print("RoBERTa Model Information:")
print("=" * 60)
print(f"Total parameters: {num_params:,}")
print(f"Trainable parameters: {num_trainable:,}")
print(f"Model size: ~{num_params * 4 / 1e6:.1f} MB (fp32)")
print("=" * 60)

### 3.3 Handle Class Imbalance

In [ ]:
# Compute class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=train_df['label'].values
)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print("Class weights for imbalance handling:")
print(f"  Ham (0): {class_weights[0]:.3f}")
print(f"  Spam (1): {class_weights[1]:.3f}")

# Custom Trainer with weighted loss
class WeightedLossTrainer(Trainer):
    """
    Custom Trainer with class-weighted loss function.
    
    Handles class imbalance by applying weights to the loss computation.
    """
    
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """
        Compute weighted cross-entropy loss.
        
        Args:
            model: The transformer model
            inputs: Dictionary with input tensors
            return_outputs: Whether to return model outputs
        
        Returns:
            Loss tensor (and optionally outputs)
        """
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Apply class weights
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

### 3.4 Training Configuration

In [ ]:
# Define compute_metrics function
def compute_metrics(eval_pred):
    """
    Compute evaluation metrics for classification.
    
    Args:
        eval_pred: EvalPrediction object with predictions and labels
    
    Returns:
        Dictionary with accuracy, precision, recall, and f1 scores
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary', pos_label=1
    )
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

# Training arguments
roberta_output_dir = MODELS_DIR / 'roberta_spam'

roberta_training_args = TrainingArguments(
    output_dir=str(roberta_output_dir),
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir=str(roberta_output_dir / 'logs'),
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    save_total_limit=2,
    seed=SEED,
    fp16=torch.cuda.is_available(),  # Mixed precision if GPU available
    gradient_checkpointing=False,  # Enable if OOM
    report_to='none',  # Disable wandb/tensorboard
)

print("Training Configuration:")
print("=" * 60)
print(f"Batch size (train): {roberta_training_args.per_device_train_batch_size}")
print(f"Batch size (eval): {roberta_training_args.per_device_eval_batch_size}")
print(f"Learning rate: {roberta_training_args.learning_rate}")
print(f"Epochs: {roberta_training_args.num_train_epochs}")
print(f"Warmup ratio: {roberta_training_args.warmup_ratio}")
print(f"Weight decay: {roberta_training_args.weight_decay}")
print(f"Mixed precision (fp16): {roberta_training_args.fp16}")
print("=" * 60)

### 3.5 Train RoBERTa Model

In [ ]:
import time

# Initialize trainer
roberta_trainer = WeightedLossTrainer(
    model=roberta_model,
    args=roberta_training_args,
    train_dataset=roberta_train,
    eval_dataset=roberta_val,
    compute_metrics=compute_metrics,
    class_weights=class_weights_tensor,
)

# Train model
print("Starting RoBERTa training...\n")
roberta_start_time = time.time()

roberta_train_result = roberta_trainer.train()

roberta_train_time = time.time() - roberta_start_time

print(f"\nTraining completed in {roberta_train_time / 60:.2f} minutes")
print(f"Final training loss: {roberta_train_result.training_loss:.4f}")

### 3.6 Evaluate RoBERTa on Test Set

In [ ]:
# Evaluate on test set
print("Evaluating RoBERTa on test set...\n")
roberta_eval_start = time.time()

roberta_test_results = roberta_trainer.evaluate(roberta_test)

roberta_inference_time = time.time() - roberta_eval_start

# Get predictions for detailed analysis
roberta_predictions = roberta_trainer.predict(roberta_test)
roberta_pred_labels = np.argmax(roberta_predictions.predictions, axis=1)
roberta_true_labels = roberta_predictions.label_ids

# Calculate per-class metrics
metrics_result = precision_recall_fscore_support(
    roberta_true_labels, roberta_pred_labels, labels=[0, 1], average=None
)
roberta_precision, roberta_recall, roberta_f1, _ = metrics_result

print("RoBERTa Test Results:")
print("=" * 60)
print(f"Overall Accuracy: {roberta_test_results['eval_accuracy']:.4f}")
print()
print("Per-class metrics:")
print(f"  Ham (0):")
print(f"    Precision: {roberta_precision[0]:.4f}")
print(f"    Recall:    {roberta_recall[0]:.4f}")
print(f"    F1-score:  {roberta_f1[0]:.4f}")
print()
print(f"  Spam (1):")
print(f"    Precision: {roberta_precision[1]:.4f}")
print(f"    Recall:    {roberta_recall[1]:.4f}")
print(f"    F1-score:  {roberta_f1[1]:.4f}")
print()
print(f"Inference time: {roberta_inference_time:.2f} seconds")
throughput = len(roberta_test) / roberta_inference_time
print(f"Throughput: {throughput:.1f} samples/sec")
print("=" * 60)

# Confusion matrix
roberta_cm = confusion_matrix(roberta_true_labels, roberta_pred_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(
    roberta_cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Ham', 'Spam'],
    yticklabels=['Ham', 'Spam']
)
plt.title('RoBERTa Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# Classification report
print("\nDetailed Classification Report:")
print(classification_report(
    roberta_true_labels,
    roberta_pred_labels,
    target_names=['Ham', 'Spam']
))

### 3.7 Save RoBERTa Model

In [ ]:
# Save model and tokenizer
roberta_trainer.save_model(str(roberta_output_dir / 'best_model'))
roberta_tokenizer.save_pretrained(str(roberta_output_dir / 'best_model'))

# Save training history
roberta_history = {
    'train_time_minutes': roberta_train_time / 60,
    'final_train_loss': roberta_train_result.training_loss,
    'test_metrics': {
        'accuracy': float(roberta_test_results['eval_accuracy']),
        'precision_spam': float(roberta_precision[1]),
        'recall_spam': float(roberta_recall[1]),
        'f1_spam': float(roberta_f1[1]),
        'precision_ham': float(roberta_precision[0]),
        'recall_ham': float(roberta_recall[0]),
        'f1_ham': float(roberta_f1[0]),
    },
    'inference_time_seconds': roberta_inference_time,
    'throughput_samples_per_sec': len(roberta_test) / roberta_inference_time,
    'confusion_matrix': roberta_cm.tolist(),
}

with open(roberta_output_dir / 'training_history.json', 'w') as f:
    json.dump(roberta_history, f, indent=2)

print(f"Model saved to: {roberta_output_dir / 'best_model'}")
print(f"Training history saved to: {roberta_output_dir / 'training_history.json'}")

## 4. DistilBERT Fine-tuning

DistilBERT is a distilled version of BERT that is 40% smaller and 60% faster
while retaining 97% of BERT's language understanding capabilities.

### 4.1 Tokenizer and Dataset Preparation

In [ ]:
# Load DistilBERT tokenizer
distilbert_tokenizer = DistilBertTokenizer.from_pretrained(
    'distilbert-base-uncased'
)

# Tokenize datasets
distilbert_train = train_dataset.map(
    lambda x: tokenize_function(x, distilbert_tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=['text']
)
distilbert_val = val_dataset.map(
    lambda x: tokenize_function(x, distilbert_tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=['text']
)
distilbert_test = test_dataset.map(
    lambda x: tokenize_function(x, distilbert_tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=['text']
)

# Set format for PyTorch
format_cols = ['input_ids', 'attention_mask', 'label']
distilbert_train.set_format(type='torch', columns=format_cols)
distilbert_val.set_format(type='torch', columns=format_cols)
distilbert_test.set_format(type='torch', columns=format_cols)

print(f"DistilBERT tokenized datasets:")
print(f"  Train: {len(distilbert_train)} samples")
print(f"  Validation: {len(distilbert_val)} samples")
print(f"  Test: {len(distilbert_test)} samples")

### 4.2 Model Initialization

In [ ]:
# Load DistilBERT model for sequence classification
distilbert_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2,
    problem_type='single_label_classification'
)

# Move to device
distilbert_model.to(device)

# Display model info
num_params_distil = sum(p.numel() for p in distilbert_model.parameters())
trainable_distil = distilbert_model.parameters()
num_trainable_distil = sum(p.numel() for p in trainable_distil if p.requires_grad)

print("DistilBERT Model Information:")
print("=" * 60)
print(f"Total parameters: {num_params_distil:,}")
print(f"Trainable parameters: {num_trainable_distil:,}")
print(f"Model size: ~{num_params_distil * 4 / 1e6:.1f} MB (fp32)")
param_reduction = (1 - num_params_distil / num_params) * 100
print(f"\nParameter reduction vs RoBERTa: {param_reduction:.1f}%")
print("=" * 60)

### 4.3 Training Configuration

In [ ]:
# Training arguments
distilbert_output_dir = MODELS_DIR / 'distilbert_spam'

distilbert_training_args = TrainingArguments(
    output_dir=str(distilbert_output_dir),
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir=str(distilbert_output_dir / 'logs'),
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    save_total_limit=2,
    seed=SEED,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=False,
    report_to='none',
)

print("DistilBERT Training Configuration:")
print("=" * 60)
print(f"Batch size (train): {distilbert_training_args.per_device_train_batch_size}")
print(f"Batch size (eval): {distilbert_training_args.per_device_eval_batch_size}")
print(f"Learning rate: {distilbert_training_args.learning_rate}")
print(f"Epochs: {distilbert_training_args.num_train_epochs}")
print(f"Warmup ratio: {distilbert_training_args.warmup_ratio}")
print(f"Weight decay: {distilbert_training_args.weight_decay}")
print(f"Mixed precision (fp16): {distilbert_training_args.fp16}")
print("=" * 60)

### 4.4 Train DistilBERT Model

In [ ]:
# Initialize trainer
distilbert_trainer = WeightedLossTrainer(
    model=distilbert_model,
    args=distilbert_training_args,
    train_dataset=distilbert_train,
    eval_dataset=distilbert_val,
    compute_metrics=compute_metrics,
    class_weights=class_weights_tensor,
)

# Train model
print("Starting DistilBERT training...\n")
distilbert_start_time = time.time()

distilbert_train_result = distilbert_trainer.train()

distilbert_train_time = time.time() - distilbert_start_time

print(f"\nTraining completed in {distilbert_train_time / 60:.2f} minutes")
print(f"Final training loss: {distilbert_train_result.training_loss:.4f}")
print(f"\nSpeedup vs RoBERTa: {roberta_train_time / distilbert_train_time:.2f}x")

### 4.5 Evaluate DistilBERT on Test Set

In [ ]:
# Evaluate on test set
print("Evaluating DistilBERT on test set...\n")
distilbert_eval_start = time.time()

distilbert_test_results = distilbert_trainer.evaluate(distilbert_test)

distilbert_inference_time = time.time() - distilbert_eval_start

# Get predictions for detailed analysis
distilbert_predictions = distilbert_trainer.predict(distilbert_test)
distilbert_pred_labels = np.argmax(distilbert_predictions.predictions, axis=1)
distilbert_true_labels = distilbert_predictions.label_ids

# Calculate per-class metrics
metrics_result = precision_recall_fscore_support(
    distilbert_true_labels, distilbert_pred_labels, labels=[0, 1], average=None
)
distilbert_precision, distilbert_recall, distilbert_f1, _ = metrics_result

print("DistilBERT Test Results:")
print("=" * 60)
print(f"Overall Accuracy: {distilbert_test_results['eval_accuracy']:.4f}")
print()
print("Per-class metrics:")
print(f"  Ham (0):")
print(f"    Precision: {distilbert_precision[0]:.4f}")
print(f"    Recall:    {distilbert_recall[0]:.4f}")
print(f"    F1-score:  {distilbert_f1[0]:.4f}")
print()
print(f"  Spam (1):")
print(f"    Precision: {distilbert_precision[1]:.4f}")
print(f"    Recall:    {distilbert_recall[1]:.4f}")
print(f"    F1-score:  {distilbert_f1[1]:.4f}")
print()
print(f"Inference time: {distilbert_inference_time:.2f} seconds")
throughput = len(distilbert_test) / distilbert_inference_time
print(f"Throughput: {throughput:.1f} samples/sec")
speedup = roberta_inference_time / distilbert_inference_time
print(f"\nSpeedup vs RoBERTa: {speedup:.2f}x")
print("=" * 60)

# Confusion matrix
distilbert_cm = confusion_matrix(distilbert_true_labels, distilbert_pred_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(
    distilbert_cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Ham', 'Spam'],
    yticklabels=['Ham', 'Spam']
)
plt.title('DistilBERT Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# Classification report
print("\nDetailed Classification Report:")
print(classification_report(
    distilbert_true_labels,
    distilbert_pred_labels,
    target_names=['Ham', 'Spam']
))

### 4.6 Save DistilBERT Model

In [ ]:
# Save model and tokenizer
distilbert_trainer.save_model(str(distilbert_output_dir / 'best_model'))
distilbert_tokenizer.save_pretrained(str(distilbert_output_dir / 'best_model'))

# Save training history
distilbert_history = {
    'train_time_minutes': distilbert_train_time / 60,
    'final_train_loss': distilbert_train_result.training_loss,
    'test_metrics': {
        'accuracy': float(distilbert_test_results['eval_accuracy']),
        'precision_spam': float(distilbert_precision[1]),
        'recall_spam': float(distilbert_recall[1]),
        'f1_spam': float(distilbert_f1[1]),
        'precision_ham': float(distilbert_precision[0]),
        'recall_ham': float(distilbert_recall[0]),
        'f1_ham': float(distilbert_f1[0]),
    },
    'inference_time_seconds': distilbert_inference_time,
    'throughput_samples_per_sec': len(distilbert_test) / distilbert_inference_time,
    'confusion_matrix': distilbert_cm.tolist(),
}

with open(distilbert_output_dir / 'training_history.json', 'w') as f:
    json.dump(distilbert_history, f, indent=2)

print(f"Model saved to: {distilbert_output_dir / 'best_model'}")
print(f"Training history saved to: {distilbert_output_dir / 'training_history.json'}")

## 5. Model Comparison

Compare RoBERTa and DistilBERT on performance, efficiency, and speed.

In [ ]:
# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Metric': [
        'Parameters',
        'Model Size (MB)',
        'Training Time (min)',
        'Inference Time (sec)',
        'Throughput (samples/sec)',
        'Accuracy',
        'Precision (Spam)',
        'Recall (Spam)',
        'F1 (Spam)',
        'Precision (Ham)',
        'Recall (Ham)',
        'F1 (Ham)',
    ],
    'RoBERTa': [
        f"{num_params:,}",
        f"{num_params * 4 / 1e6:.1f}",
        f"{roberta_train_time / 60:.2f}",
        f"{roberta_inference_time:.2f}",
        f"{len(roberta_test) / roberta_inference_time:.1f}",
        f"{roberta_test_results['eval_accuracy']:.4f}",
        f"{roberta_precision[1]:.4f}",
        f"{roberta_recall[1]:.4f}",
        f"{roberta_f1[1]:.4f}",
        f"{roberta_precision[0]:.4f}",
        f"{roberta_recall[0]:.4f}",
        f"{roberta_f1[0]:.4f}",
    ],
    'DistilBERT': [
        f"{num_params_distil:,}",
        f"{num_params_distil * 4 / 1e6:.1f}",
        f"{distilbert_train_time / 60:.2f}",
        f"{distilbert_inference_time:.2f}",
        f"{len(distilbert_test) / distilbert_inference_time:.1f}",
        f"{distilbert_test_results['eval_accuracy']:.4f}",
        f"{distilbert_precision[1]:.4f}",
        f"{distilbert_recall[1]:.4f}",
        f"{distilbert_f1[1]:.4f}",
        f"{distilbert_precision[0]:.4f}",
        f"{distilbert_recall[0]:.4f}",
        f"{distilbert_f1[0]:.4f}",
    ],
})

print("\n" + "=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

### Performance Visualization

In [ ]:
# Prepare data for visualization
metrics = ['Accuracy', 'Precision\n(Spam)', 'Recall\n(Spam)', 'F1\n(Spam)']
roberta_values = [
    roberta_test_results['eval_accuracy'],
    roberta_precision[1],
    roberta_recall[1],
    roberta_f1[1],
]
distilbert_values = [
    distilbert_test_results['eval_accuracy'],
    distilbert_precision[1],
    distilbert_recall[1],
    distilbert_f1[1],
]

# Create comparison plots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Performance metrics
x = np.arange(len(metrics))
width = 0.35

roberta_color = '#1f77b4'
distilbert_color = '#ff7f0e'
axes[0].bar(x - width/2, roberta_values, width, label='RoBERTa', alpha=0.8)
bar_distil = axes[0].bar(
    x + width/2, distilbert_values, width, label='DistilBERT', alpha=0.8
)
axes[0].set_ylabel('Score')
axes[0].set_title('Performance Metrics Comparison')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].set_ylim([0.9, 1.0])  # Focus on high-performance range
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: Training time comparison
time_comparison = [
    roberta_train_time / 60,
    distilbert_train_time / 60
]
bar_colors = [roberta_color, distilbert_color]
axes[1].bar(['RoBERTa', 'DistilBERT'], time_comparison, alpha=0.8, color=bar_colors)
axes[1].set_ylabel('Time (minutes)')
axes[1].set_title('Training Time Comparison')
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(time_comparison):
    axes[1].text(i, v + 0.1, f"{v:.2f}", ha='center', va='bottom')

# Plot 3: Inference throughput comparison
throughput_comparison = [
    len(roberta_test) / roberta_inference_time,
    len(distilbert_test) / distilbert_inference_time
]
axes[2].bar(
    ['RoBERTa', 'DistilBERT'], throughput_comparison, alpha=0.8, color=bar_colors
)
axes[2].set_ylabel('Samples/second')
axes[2].set_title('Inference Throughput Comparison')
axes[2].grid(axis='y', alpha=0.3)
for i, v in enumerate(throughput_comparison):
    axes[2].text(i, v + 1, f"{v:.1f}", ha='center', va='bottom')

plt.tight_layout()
plt.show()

### Speed vs Accuracy Trade-off

In [ ]:
# Calculate speedup and accuracy difference
train_speedup = roberta_train_time / distilbert_train_time
inference_speedup = roberta_inference_time / distilbert_inference_time
accuracy_diff = (
    roberta_test_results['eval_accuracy'] 
    - distilbert_test_results['eval_accuracy']
)
precision_diff = roberta_precision[1] - distilbert_precision[1]
f1_diff = roberta_f1[1] - distilbert_f1[1]

print("\n" + "=" * 80)
print("SPEED VS ACCURACY TRADE-OFF ANALYSIS")
print("=" * 80)
print(f"\nDistilBERT vs RoBERTa:")
print(f"  Training speedup:   {train_speedup:.2f}x faster")
print(f"  Inference speedup:  {inference_speedup:.2f}x faster")
size_reduction = (1 - num_params_distil / num_params) * 100
print(f"  Model size:         {size_reduction:.1f}% smaller")
print()
accuracy_pct = accuracy_diff * 100
print(f"  Accuracy difference:       {accuracy_diff:+.4f} ({accuracy_pct:+.2f}%)")
precision_pct = precision_diff * 100
print(f"  Precision (Spam) diff:     {precision_diff:+.4f} ({precision_pct:+.2f}%)")
f1_pct = f1_diff * 100
print(f"  F1 (Spam) diff:            {f1_diff:+.4f} ({f1_pct:+.2f}%)")
print("=" * 80)

## 6. Summary and Recommendations

### Key Observations

**RoBERTa:**
- Larger model with ~125M parameters
- Superior performance metrics, particularly on precision
- Higher computational cost (training and inference)
- Best choice when accuracy is paramount

**DistilBERT:**
- Lightweight model with ~66M parameters (40% smaller)
- Competitive performance with minor accuracy trade-off
- Significantly faster training and inference
- Excellent choice for resource-constrained environments

### Model Selection Criteria

**Choose RoBERTa if:**
- Maximum accuracy and precision are critical
- GPU resources are available
- Inference latency is not a primary concern
- Cost of false positives is very high

**Choose DistilBERT if:**
- Fast inference is required (e.g., real-time applications)
- Limited computational resources (CPU deployment)
- Model size matters (edge devices, mobile)
- Slight accuracy trade-off is acceptable

### Production Recommendations

1. **For high-volume production systems**: Deploy DistilBERT for faster
   throughput, potentially with quantization for additional speedup

2. **For critical filtering**: Use RoBERTa to minimize false positives

3. **Hybrid approach**: Use DistilBERT for initial filtering, RoBERTa
   for borderline cases (confidence-based routing)

4. **Model optimization**: Consider:
   - ONNX conversion for faster inference
   - Quantization (INT8) for 4x speedup
   - Knowledge distillation for custom lightweight models

5. **Monitoring**: Track precision on spam class in production to ensure
   legitimate messages are not filtered

In [ ]:
# Save comparison results
comparison_results = {
    'roberta': roberta_history,
    'distilbert': distilbert_history,
    'comparison': {
        'train_speedup': float(train_speedup),
        'inference_speedup': float(inference_speedup),
        'size_reduction_percent': float((1 - num_params_distil / num_params) * 100),
        'accuracy_difference': float(accuracy_diff),
        'precision_spam_difference': float(precision_diff),
        'f1_spam_difference': float(f1_diff),
    },
    'recommendation': 'RoBERTa for maximum precision, DistilBERT for production speed',
}

with open(MODELS_DIR / 'model_comparison.json', 'w') as f:
    json.dump(comparison_results, f, indent=2)

print(f"\nComparison results saved to: {MODELS_DIR / 'model_comparison.json'}")
print("\nNotebook execution complete!")